# SWG attention — train the navigation query only

Freeze SparseWalker, concept keys, router, and HNSW topology. Train only a 64→16 query projection so the current working-memory hidden state points toward the next item's routed concepts. Then rerun 4-hop HNSW navigation diagnostics.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, shutil
REPO='/content/Sparsewalker'
BRANCH='agent/walker-swg-query-training'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','faiss-cpu'],check=True)
SRC=f'{REPO}/src'
env=os.environ.copy(); env['PYTHONPATH']=SRC; env['PYTHONUNBUFFERED']='1'
cmd=[sys.executable,'-u',f'{REPO}/experiments/run_swg_query_training.py',
     '--seed','42','--epochs','5','--train-cap','300000','--per-bucket','128',
     '--hnsw-m','8','--ef-construction','160']
print('RUNNING', ' '.join(cmd), flush=True)
p=subprocess.Popen(cmd,cwd=REPO,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in p.stdout:
    print(line,end='',flush=True)
rc=p.wait()
if rc: raise subprocess.CalledProcessError(rc,cmd)


## Recover a completed run without rerunning

If the launcher previously returned `CompletedProcess(... returncode=0)` but hid stdout, run this cell. The experiment writes its full result to Drive.

In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_swg_query/result.json')
assert p.exists(), f'No saved result at {p}'
r=json.loads(p.read_text())
pre=r['pretrain']['navigation']['4']
post=r['posttrain']['navigation']['4']
oracle=r['oracle_next']['4']
print('PRETRAIN_PANEL', {'dense_target_hit@10':r['pretrain']['dense_target_hit@10'],'hop4':pre})
for row in r['history']: print('QUERY_TRAIN',row)
print('POSTTRAIN_PANEL', {'dense_target_hit@10':r['posttrain']['dense_target_hit@10'],'hop4':post,'oracle_next_hop4':oracle})
print('DECISION', {
    'pre_next_seen_h4':pre['next_item_concept_seen_rate'],
    'post_next_seen_h4':post['next_item_concept_seen_rate'],
    'absolute_gain':post['next_item_concept_seen_rate']-pre['next_item_concept_seen_rate'],
    'dense_target_hit@10':r['posttrain']['dense_target_hit@10'],
    'oracle_next_h4':oracle['any_target_hit_rate']})


### Send back these lines

`PRETRAIN_PANEL`, all `QUERY_TRAIN` lines, `POSTTRAIN_PANEL`, and `DECISION`. The key metric is next-item concept seen within 4 hops; the previous untrained-query run was ~0.5% while oracle reachability was ~99.5%.